# 🌌 KANX-Studio: 生产级 KAN 网络工程与诊断平台

本工具包基于 2026 年最新发布的 `KANX` 库构建，旨在解决学术界和工业界在应用 Kolmogorov-Arnold Networks (KAN) 时的两大痛点：
1. **模型选型盲区**：通过元学习（Meta-Learning）智能诊断，打破“唯MSE论”，结合延迟开销给出四象限部署建议。
2. **生产落地困难**：提供一键式的 ONNX 导出、FastAPI 封装及 Docker 容器化方案。

In [2]:
import numpy as np
import pandas as pd
import tensorflow as tf

from kanx import KAN
from kanx_studio import KANDiagnoser, ModelDeployer

print("✅ 环境初始化成功！KANX-Studio 已加载。")

✅ 环境初始化成功！KANX-Studio 已加载。


## 模块一：🧠 智能适用性诊断 (KANDiagnoser)
我们将模拟一个极其平滑、低维度的回归数据集（最适合 KAN 的场景），并测试诊断器是否能给出正确的“强烈推荐 KAN”的建议。

In [3]:
import numpy as np

# 1. 生成真正高度平滑的低维数据 (比如时间和其正弦波，完美符合 KAN 的偏好)
t = np.linspace(0, 10, 1000)
x1 = np.sin(t)
x2 = np.cos(t)
# 特征维度 D=2，且极其平滑
X_truly_smooth = np.column_stack([x1, x2])

# 2. 实例化诊断大脑
diagnoser = KANDiagnoser(model_path="kanx_studio_diagnoser.json")

# 3. 运行极速诊断
label, status, features_extracted = diagnoser.evaluate(X_truly_smooth, task_type="regression")

print("="*50)
print(f"📊 提取的元特征: 维度={features_extracted['n_features']}, 平滑度={features_extracted['smoothness_proxy']:.6f}")
print(f"🎯 诊断结论: {status['status']}")
print(f"💡 详细建议: {status['reason']}")
print("="*50)

📊 提取的元特征: 维度=2, 平滑度=0.000049
🎯 诊断结论: 强烈推荐 KAN
💡 详细建议: 精度更高，且延迟在可接受范围内 (<= 5倍)。


## 模块二：🚀 生产部署工具链 (ModelDeployer)
在此模块中，我们将快速训练一个真实的 KAN 模型，并调用 Deployer 模块，一键生成 `ONNX` 权重文件、`app.py` 接口文件和 `Dockerfile`。

In [5]:
import tensorflow as tf

print("正在构建并训练 KAN 模型...")
model = tf.keras.Sequential([
    # 注意：我们的 X_truly_smooth 只有 2 维，所以输入维度改成 2
    KAN([2, 16, 1])
])
model.build((None, 2))
model.compile(optimizer='adam', loss='mse')

# 训练一下
model.fit(X_truly_smooth, np.random.rand(1000, 1), epochs=1, batch_size=128, verbose=0)
print("✅ 模型训练完成！")

# 启动一键部署引擎
deployer = ModelDeployer(output_dir="deployment_package")
deployer.export_onnx(model, model_name="my_kan_model")
deployer.generate_api_and_docker(model_name="my_kan_model")

正在构建并训练 KAN 模型...
✅ 模型训练完成！


✅ ONNX 模型已导出至: deployment_package\my_kan_model.onnx
✅ FastAPI (app.py) 和 Dockerfile 已成功生成！
